In [1]:
!pip install torch
!pip install transformers
!pip install tqdm

In [120]:
import torch
from torch import nn
from transformers import BertTokenizer, BertModel
from torch.optim import AdamW  
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from tqdm import tqdm

In [121]:
torch.manual_seed(42)
    
device = (
    "mps" 
    if torch.backends.mps.is_available() 
    else "cuda" 
    if torch.cuda.is_available() 
    else "cpu"
)
device = torch.device(device)
print(f"Using device: {device}")

Using device: cuda


In [122]:
sentences = "C:/Users/zhoum/Downloads/DSA4265_Project/Results/sentence_data/sentences.csv"

In [123]:
df= pd.read_csv("C:/Users/zhoum/Downloads/labeled sentences.csv")
df = df.dropna(how='all')
df = df.reset_index(drop=True)
df = df[["sentence", "sentiment", "topic"]]
df.head()
df.tail(1)

,sentence,sentiment,topic
660,"And I think in the release, it talks about 500...",Neutral,Outlook


In [124]:
sentiment_map = {
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2
}

df["sentiment_label"] = df["sentiment"].map(sentiment_map)

In [125]:
topics = ["Growth", "Risk", "Outlook", "Operation","General"]

topic_map = {"Growth":0, "Risk":1, "Outlook":2, "Operation":3, "General":4}
df["topic_label"] = df["topic"].map(topic_map)

In [126]:
df.shape

(661, 5)

In [127]:
from datasets import Dataset
from transformers import AutoTokenizer

dataset = Dataset.from_pandas(df)

model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(example):
    return tokenizer(example["sentence"], truncation=True, padding="max_length")

dataset = dataset.map(tokenize)


Map:   0%|          | 0/661 [00:00<?, ? examples/s]

In [128]:
dataset

Dataset({
    features: ['sentence', 'sentiment', 'topic', 'sentiment_label', 'topic_label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 661
})

In [129]:
import torch
import torch.nn as nn
from transformers import AutoModel
class MultiTaskModel(nn.Module):
    def __init__(self,model_name):
        super().__init__()
        self.bert=AutoModel.from_pretrained(model_name)
        hidden=self.bert.config.hidden_size

        self.sentiment_head=nn.Linear(hidden,3)
        self.topic_head=nn.Linear(hidden,5)

    def forward(self,input_ids,attention_mask,sentiment_label=None,topic_label=None):
        outputs=self.bert(input_ids=input_ids,attention_mask=attention_mask)
        pooled=outputs.pooler_output

        sentiment_logits=self.sentiment_head(pooled)
        topic_logits=self.topic_head(pooled)

        loss=None
        if sentiment_label is not None:
            ce_loss=nn.CrossEntropyLoss()(sentiment_logits,sentiment_label)
            topic_loss = nn.CrossEntropyLoss()(topic_logits, topic_label)
            loss=ce_loss+topic_loss
        return{
            "loss":loss,
            "sentiment_logits":sentiment_logits,
            "topic_logits":topic_logits
        }

In [130]:
from transformers import TrainingArguments, Trainer

def collate_fn(batch):
    return {
        "input_ids": torch.tensor([x["input_ids"] for x in batch]),
        "attention_mask": torch.tensor([x["attention_mask"] for x in batch]),
        "sentiment_label": torch.tensor([x["sentiment_label"] for x in batch]),
        "topic_label": torch.tensor([x["topic_label"] for x in batch]),

    }
model=MultiTaskModel(model_name)

training_args=TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-5
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=collate_fn
)

trainer.train()



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 
classifier.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss


TrainOutput(global_step=249, training_loss=1.2841109310288028, metrics={'train_runtime': 82.7462, 'train_samples_per_second': 23.965, 'train_steps_per_second': 3.009, 'total_flos': 0.0, 'train_loss': 1.2841109310288028, 'epoch': 3.0})

In [131]:
import torch.nn.functional as F

def predict(text):
    model.eval()
    
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs.pop("token_type_ids", None)

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    sent_probs = torch.softmax(outputs["sentiment_logits"], dim=1)
    sent_conf, sent_idx = torch.max(sent_probs, dim=1)

    topic_probs = torch.softmax(outputs["topic_logits"], dim=1)
    topic_conf, topic_idx = torch.max(topic_probs, dim=1)

    sentiment_inv = {0: "Negative", 1: "Neutral", 2: "Positive"}
    topics = ["Growth", "Risk", "Outlook", "Operation","General"]

    return {
        "sentiment": sentiment_inv[sent_idx.item()],
        "sent_conf": sent_conf.item(),
        "topic": topics[topic_idx.item()],
        "topic_conf": topic_conf.item()
    }

In [132]:
predict("The revenue grew ten-fold")

{'sentiment': 'Positive',
 'sent_conf': 0.9394201636314392,
 'topic': 'Growth',
 'topic_conf': 0.8687777519226074}

In [133]:
df_all = pd.read_csv("C:/Users/zhoum/Downloads/DSA4265_Project/Results/sentence_data/sentences.csv")
df_all = df_all[df_all['company'] != 'unknown']
df_labeled = pd.read_csv("C:/Users/zhoum/Downloads/labeled sentences.csv")
df_labeled = df_labeled.dropna(how='all')

In [135]:
df_all.tail()

,company,quarter,year,company_quarter,turn_id,sentence_id,speaker,sentence
3534,tesla,Q4,2025,tesla_Q4_2025,72,5,Elon Musk,"We're pretty much the not just the largest, bu..."
3535,tesla,Q4,2025,tesla_Q4_2025,72,6,Elon Musk,We're making moves to make sure that no matter...
3536,tesla,Q4,2025,tesla_Q4_2025,73,1,Travis Axelrod,"Unfortunately, that's all the time we have for..."
3537,tesla,Q4,2025,tesla_Q4_2025,73,2,Travis Axelrod,"We really appreciate everyone's questions, and..."
3538,tesla,Q4,2025,tesla_Q4_2025,73,3,Travis Axelrod,"Thank you very much, and goodbye."


In [136]:
df_unlabeled = df_all[~df_all["sentence"].isin(df_labeled["sentence"])].copy()
df_unlabeled.shape

(2872, 8)

In [137]:

results=[]
for text in tqdm(df_unlabeled["sentence"]):
    pred = predict(text)
    results.append(pred)



100%|██████████████████████████████████████████████████████████████████████████████| 2872/2872 [01:05<00:00, 44.11it/s]


In [140]:
df_unlabeled = df_unlabeled.reset_index(drop=True)
df_preds = pd.DataFrame(results)

print(len(df_unlabeled), len(df_preds))

2872 2872


In [141]:

df_unlabeled = pd.concat([df_unlabeled, df_preds], axis=1)
print(df_unlabeled.shape)
print(df_preds.shape)
print(df_unlabeled.tail(10))

(2872, 12)
(2872, 4)
     company quarter  year company_quarter  turn_id  sentence_id  \
2862   tesla      Q4  2025   tesla_Q4_2025       72            0   
2863   tesla      Q4  2025   tesla_Q4_2025       72            1   
2864   tesla      Q4  2025   tesla_Q4_2025       72            2   
2865   tesla      Q4  2025   tesla_Q4_2025       72            3   
2866   tesla      Q4  2025   tesla_Q4_2025       72            4   
2867   tesla      Q4  2025   tesla_Q4_2025       72            5   
2868   tesla      Q4  2025   tesla_Q4_2025       72            6   
2869   tesla      Q4  2025   tesla_Q4_2025       73            1   
2870   tesla      Q4  2025   tesla_Q4_2025       73            2   
2871   tesla      Q4  2025   tesla_Q4_2025       73            3   

             speaker                                           sentence  \
2862       Elon Musk              Why do we have to build these things?   
2863       Elon Musk  Why can others not also please build these thi...   
2864 

In [142]:
df_labeled.head()

,annotation_id,annotator,company,company_quarter,created_at,id,lead_time,quarter,sentence,sentence_id,sentiment,speaker,topic,turn_id,updated_at,year
1,1.0,1.0,amazon,amazon_Q3_2025,2026-03-25T14:17:41.370595Z,1.0,35.953,Q3,"Hello, and welcome to our Q3 2025 financial re...",0.0,Neutral,Dave Fildes,General,0.0,2026-03-25T14:17:41.370624Z,2025.0
3,2.0,1.0,amazon,amazon_Q3_2025,2026-03-25T14:17:52.794252Z,2.0,10.820,Q3,Joining us today to answer your questions is A...,1.0,Neutral,Dave Fildes,General,0.0,2026-03-25T14:17:52.794272Z,2025.0
5,3.0,1.0,amazon,amazon_Q3_2025,2026-03-25T14:18:05.923615Z,3.0,12.670,Q3,"As you listen to today's conference call, we e...",2.0,Neutral,Dave Fildes,General,0.0,2026-03-25T14:18:05.923646Z,2025.0
7,4.0,1.0,amazon,amazon_Q3_2025,2026-03-25T14:18:17.943894Z,4.0,11.542,Q3,"Please note, unless otherwise stated.",3.0,Neutral,Dave Fildes,General,0.0,2026-03-25T14:18:17.943924Z,2025.0
9,5.0,1.0,amazon,amazon_Q3_2025,2026-03-25T14:18:25.249773Z,5.0,6.884,Q3,All comparisons in this call will be against o...,4.0,Neutral,Dave Fildes,General,0.0,2026-03-25T14:18:25.249803Z,2025.0


In [144]:

df_unlabeled_clean = df_unlabeled[['company', 'quarter', 'year', 'company_quarter',
                                   'turn_id', 'sentence_id', 'speaker', 'sentence',
                                   'sentiment', 'sent_conf', 'topic', 'topic_conf']].copy()


df_labeled_clean = df_labeled[['company', 'quarter', 'year', 'company_quarter',
                               'turn_id', 'sentence_id', 'speaker', 'sentence',
                               'sentiment', 'topic']].copy()

df_labeled_clean['sent_conf'] = None
df_labeled_clean['topic_conf'] = None


df_full = pd.concat([df_labeled_clean, df_unlabeled_clean], ignore_index=True)


df_full = df_full.drop_duplicates(subset=['sentence'])


df_full.to_csv(r"C:/Users/zhoum/Downloads/DSA4265_Project/Results/df_full_combined.csv", index=False)

print("Combined CSV saved. Shape:", df_full.shape)
print(df_full.head())


Combined CSV saved. Shape: (3462, 12)
  company quarter    year company_quarter  turn_id  sentence_id      speaker  \
0  amazon      Q3  2025.0  amazon_Q3_2025      0.0          0.0  Dave Fildes   
1  amazon      Q3  2025.0  amazon_Q3_2025      0.0          1.0  Dave Fildes   
2  amazon      Q3  2025.0  amazon_Q3_2025      0.0          2.0  Dave Fildes   
3  amazon      Q3  2025.0  amazon_Q3_2025      0.0          3.0  Dave Fildes   
4  amazon      Q3  2025.0  amazon_Q3_2025      0.0          4.0  Dave Fildes   

                                            sentence sentiment    topic  \
0  Hello, and welcome to our Q3 2025 financial re...   Neutral  General   
1  Joining us today to answer your questions is A...   Neutral  General   
2  As you listen to today's conference call, we e...   Neutral  General   
3              Please note, unless otherwise stated.   Neutral  General   
4  All comparisons in this call will be against o...   Neutral  General   

  sent_conf topic_conf  
0    

In [145]:
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset as TorchDataset, DataLoader



df_full["sentiment_label"] = df_full["sentiment"].map(sentiment_map)

df_full["topic_label"] = df_full["topic"].map(topic_map)
df_full["year"] = pd.to_numeric(df_full["year"], errors="coerce") 

dataset_full = Dataset.from_pandas(df_full)

model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_batch(batch):
    return tokenizer(
        batch["sentence"],
        truncation=True,
        padding="max_length",
        max_length=128,     
    )


dataset_full = dataset_full.map(tokenize_batch, batched=True)

split_dataset = dataset_full.train_test_split(test_size=0.2, seed=42)
train_hf = split_dataset["train"]
test_hf = split_dataset["test"]


class MultiTaskDataset(TorchDataset):
    def __init__(self, hf_dataset):
        self.dataset = hf_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "sentiment_label": torch.tensor(item["sentiment_label"], dtype=torch.long),
            "topic_label": torch.tensor(item["topic_label"], dtype=torch.long),
        }


train_dataset = MultiTaskDataset(train_hf)
test_dataset = MultiTaskDataset(test_hf)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)



Map:   0%|          | 0/3462 [00:00<?, ? examples/s]

In [146]:
for i in range(len(train_dataset)):
    item = train_dataset.dataset[i]  # underlying HF dataset
    if item["sentiment_label"] is None or item["topic_label"] is None:
        print(f"Missing labels at index {i}: {item}")

In [147]:
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 3
num_training_steps = num_epochs * len(train_loader)
scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

# ----- Training Loop -----
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for batch in progress_bar:
        optimizer.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        sentiment_labels = batch["sentiment_label"].to(device)
        topic_labels = batch["topic_label"].to(device)

        outputs = model(input_ids, attention_mask, sentiment_labels, topic_labels)
        loss = outputs["loss"]

        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        progress_bar.set_postfix({"loss": f"{total_loss / (progress_bar.n + 1):.4f}"})

    print(f"Epoch {epoch+1} finished | Avg Train Loss: {total_loss / len(train_loader):.4f}")

# ----- Evaluation -----
model.eval()
correct_sentiment, correct_topic, total = 0, 0, 0

with torch.no_grad():
    progress_bar = tqdm(test_loader, desc="Evaluating")
    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        sentiment_labels = batch["sentiment_label"].to(device)
        topic_labels = batch["topic_label"].to(device)

        outputs = model(input_ids, attention_mask)
        sentiment_logits = outputs["sentiment_logits"]
        topic_logits = outputs["topic_logits"]

        correct_sentiment += (sentiment_logits.argmax(dim=1) == sentiment_labels).sum().item()
        correct_topic += (topic_logits.argmax(dim=1) == topic_labels).sum().item()
        total += input_ids.size(0)

print(f"Sentiment Accuracy: {correct_sentiment / total:.4f}")
print(f"Topic Accuracy: {correct_topic / total:.4f}")

Epoch 1/3: 100%|████████████████████████████████████████████████████████| 174/174 [00:29<00:00,  5.86it/s, loss=0.6144]


Epoch 1 finished | Avg Train Loss: 0.6144


Epoch 2/3: 100%|████████████████████████████████████████████████████████| 174/174 [00:29<00:00,  5.87it/s, loss=0.3302]


Epoch 2 finished | Avg Train Loss: 0.3302


Epoch 3/3: 100%|████████████████████████████████████████████████████████| 174/174 [00:29<00:00,  5.88it/s, loss=0.2215]


Epoch 3 finished | Avg Train Loss: 0.2215


Evaluating: 100%|██████████████████████████████████████████████████████████████████████| 44/44 [00:02<00:00, 16.55it/s]

Sentiment Accuracy: 0.9509
Topic Accuracy: 0.8860


topic_label
3    1352
2     760
4     746
0     596
1       8
Name: count, dtype: int64